## Preprocessing: Stage 2 of the pipeline

This script turns the raw scraped records into a clean, and stratified test/train split. Runs before feature engineering and only touches lightweight record metadata. 
### Roles:
1. Load and validate raw JSON records
2. Deduplicate by label and hostname: guards against the same domain being pulled in by more than one overlapping phishing feed and inflating that domain's weight in the dataset.
3. Stratified 80/20 train-test split, writing a manifest and per-split lists. Stratification alone preserves the exact class ratio in both splits.

In [1]:
# Library imports
import glob
import json
import logging
import os
import sys
import pandas as pd
from sklearn.model_selection import train_test_split

In [ ]:
# Directory mapping
RAW_DIR = "DA/raw"
PROCESSED_DIR = "DA/processed"
LOG_PATH = "logs/preprocessed.log"
manifest_path = os.path.join(PROCESSED_DIR, "dataset_manifest.csv")

os.makedirs(PROCESSED_DIR, exist_ok=True)
os.makedirs("logs", exist_ok=True)

In [3]:
# Logging configuration
stream_handler = logging.StreamHandler(sys.stdout)
stream_handler.flush = sys.stdout.flush
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    handlers=[logging.FileHandler(LOG_PATH), stream_handler],
    force=True,
)
log = logging.getLogger(__name__)

In [4]:
# Required fields (global schema)
REQUIRED_FIELDS = ["sample_id", "label", "source", "seed_url", 
                   "final_url", "hostname", "http_status", "raw_html"]

### Loading and Validation

In [5]:
progress_every = 2000 # report processing processing after every 2000
                      # processed files

paths = glob.glob(os.path.join(RAW_DIR, "*.json"))  # load records
log.info(f"Total loaded records: {len(paths)}")


2026-09-30 15:00:31,487 [INFO] Total loaded records: 30835


In [6]:
# checking for corrupt rows and records with missing fields
rows, n_corrupt, n_misisng_fieds = [], 0, 0

for i, p in enumerate(paths, start=1):
    try:
        with open(p, "r", encoding="utf-8") as f:
            rec = json.load(f)
    except (json.JSONDecodeError, UnicodeDecodeError, OSError) as e:
        log.warning(f"[corrupt] could not parse -> {e}")
        n_corrupt += 1
        continue

    missing = [field for field in REQUIRED_FIELDS if field not in rec or rec[field] in (None, "")]

    if missing:
        log.warning(f"[incomplete] {p} missing fields: {missing} -- DROPPED")
        n_misisng_fieds += 1
        continue

    rows.append({
        "sample_id": rec["sample_id"],
        "url": rec["final_url"],
        "source": rec["source"],
        "hostname": rec["hostname"],
        "http_status": rec["http_status"],
        "n_inline_scripts": len(rec.get("inline_scripts", [])),
        "n_external_scripts": len(rec.get("external_script_srcs", [])),
        "html_length": len(rec.get("raw_html", "")),
        "label": 1 if rec["label"] == "phishing" else 0
    })

    if progress_every and i % progress_every == 0:
        log.info(f"Processed {i}/{len(paths)} raw files")

log.info(
    f"Loaded {len(rows)} valid records "
    f"({n_corrupt} corrupt, {n_misisng_fieds} missing required fields discarded)"
)

2026-09-30 15:00:31,622 [WARNING] [incomplete] raw/581b9a4cdbc3c1ff.json missing fields: ['raw_html'] -- DROPPED
2026-09-30 15:00:31,921 [WARNING] [incomplete] raw/417301a73a6e6944.json missing fields: ['raw_html'] -- DROPPED
2026-09-30 15:00:32,047 [WARNING] [incomplete] raw/a3c9f45f137a4732.json missing fields: ['raw_html'] -- DROPPED
2026-09-30 15:00:32,536 [WARNING] [incomplete] raw/19428ae8c4b0fda1.json missing fields: ['raw_html'] -- DROPPED
2026-09-30 15:00:32,553 [WARNING] [incomplete] raw/056cf1f8efa9924e.json missing fields: ['raw_html'] -- DROPPED
2026-09-30 15:00:32,585 [WARNING] [incomplete] raw/e905b80b63f9646d.json missing fields: ['raw_html'] -- DROPPED
2026-09-30 15:00:32,775 [WARNING] [incomplete] raw/dcb7eea999c97956.json missing fields: ['raw_html'] -- DROPPED
2026-09-30 15:00:32,836 [INFO] Processed 2000/30835 raw files
2026-09-30 15:00:32,998 [WARNING] [incomplete] raw/47943ef421f6f510.json missing fields: ['raw_html'] -- DROPPED
2026-09-30 15:00:33,004 [WARNING] 

### Deduplication

In [7]:
df = pd.DataFrame(rows)
df.head()

,sample_id,url,source,hostname,http_status,n_inline_scripts,n_external_scripts,html_length,label
0,96c874591d229821,https://olist.com/,tranco,olist.com,200,6,33,754952,0
1,8da21e83276732d0,https://www.carecredit.com/,tranco,www.carecredit.com,200,28,9,160011,0
2,cca1dfb0c792336e,https://q-r.to/bgc03U,phishtank,q-r.to,200,1,2,1663,1
3,b92b3190dfeb0d6b,https://mainejews.org/redirect-url-for-iframe....,phishtank,mainejews.org,200,0,0,633,1
4,e544b4eeb04ecbdd,https://www.afterpay.com/global/country-select,tranco,www.afterpay.com,200,1,10,5637,0


In [8]:
dedup_df = df.drop_duplicates(subset=["label", "hostname"], keep="first").reset_index(drop=True)
log.info(
    f"Before deduplication: {len(df)} "
    f"After deduplication: {len(dedup_df)}"
    )

2026-09-30 15:00:49,985 [INFO] Before deduplication: 30664 After deduplication: 17269


In [9]:
label_counts = dedup_df["label"].value_counts()
log.info(
    f"[reporting class balance] \n{label_counts}"
)

def report_balance(df, stage_label):
    counts = df["label"].value_counts()
    log.info(f"Class label [{stage_label}]: " + ", ".join(f"{k}={v}" for k, v in counts.items()))

2026-09-30 15:00:49,989 [INFO] [reporting class balance] 
label
0    11480
1     5789
Name: count, dtype: int64


### Dataset splits

In [10]:
train_df, test_df = train_test_split(dedup_df, test_size=0.2, random_state=42,
                                     stratify=dedup_df["label"])
report_balance(train_df, "train split")
report_balance(test_df, "test split")

2026-09-30 15:00:50,003 [INFO] Class label [train split]: 0=9184, 1=4631
2026-09-30 15:00:50,004 [INFO] Class label [test split]: 0=2296, 1=1158


In [11]:
# save clean dataset as csv
train_df = train_df.assign(split="train")
test_df = test_df.assign(split="test")
full_df = pd.concat([train_df, test_df]).reset_index(drop=True)

full_df.to_csv(manifest_path, index=False)
log.info(f"Wrote dataset manifest -> {manifest_path} {len(full_df)} rows")

# write train-test ids
for name, part in [("train", train_df), ("test", test_df)]:
    ids_path = os.path.join(PROCESSED_DIR, f"{name}_ids.txt")
    with open(ids_path, "w") as f:
        f.write("\n".join(part["sample_id"].tolist()))
    log.info(f"Wrote {name} split IDs -> ids_path ({len(part)} ids)")

2026-09-30 15:00:50,034 [INFO] Wrote dataset manifest -> processed/dataset_manifest.csv 17269 rows
2026-09-30 15:00:50,035 [INFO] Wrote train split IDs -> ids_path (13815 ids)
2026-09-30 15:00:50,036 [INFO] Wrote test split IDs -> ids_path (3454 ids)
